# Notebook 6: Econometric Modelling

**Research Project:** Improving Asymmetric Exchange Rate Pass-Through Modelling Across Food Price Categories in South Africa Using Machine Learning

**Primary Modelling Period:** April 2017 – December 2025

**Unit of Analysis:** Food subclass × month

## Notebook Overview

This notebook develops the econometric component of the research using the
balanced food-subclass panel prepared in Notebook 5.

The econometric analysis has two purposes:

1. estimate a symmetric exchange-rate pass-through benchmark using ARDL models;
2. estimate asymmetric pass-through using nonlinear ARDL models that separate Rand depreciation and appreciation movements.

Models are estimated separately for each eligible food subclass. This allows the magnitude, direction and timing of exchange-rate pass-through to diffe across food categories.

The notebook covers model eligibility testing, stationarity analysis, lag selection and model estimation. Detailed diagnostics and research
interpretation will be completed in Notebook 7.

In [1]:
# import required libraries
from pathlib import Path

import pandas as pd
import numpy as np
import warnings

from statsmodels.tsa.ardl import ARDL, UECM
from statsmodels.tsa.stattools import adfuller, kpss, zivot_andrews
from statsmodels.tools.sm_exceptions import InterpolationWarning

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)


In [2]:
# load the econometric dataset
data_path = Path("../data/processed/econometric_model_data.csv")

if not data_path.exists():
    raise FileNotFoundError(
        f"Econometric dataset not found: {data_path.resolve()}"
    )

econometric_data = pd.read_csv(
    data_path,
    parse_dates=["Date"]
)

econometric_data = (
    econometric_data
    .sort_values(["SubclassDescription", "Date"])
    .reset_index(drop=True)
)

print(f"Dataset shape: {econometric_data.shape}")
print(
    "Period:",
    econometric_data["Date"].min().date(),
    "to",
    econometric_data["Date"].max().date()
)
print(
    "Food subclasses:",
    econometric_data["SubclassDescription"].nunique()
)

Dataset shape: (4830, 15)
Period: 2017-04-01 to 2025-12-01
Food subclasses: 46


In [3]:
# validate the modelling structure
required_columns = [
    "Date",
    "GroupDescription",
    "ClassDescription",
    "SubclassDescription",
    "Subclass_Weight",
    "CPI",
    "Log_CPI",
    "Food_Inflation_Pct",
    "ExchangeRate",
    "Log_ExchangeRate",
    "ExchangeRate_Log_Change_Pct",
    "Depreciation_Shock_Pct",
    "Appreciation_Shock_Pct",
    "ExchangeRate_Positive_Cumulative_Pct",
    "ExchangeRate_Negative_Cumulative_Pct"
]

missing_columns = sorted(
    set(required_columns) - set(econometric_data.columns)
)

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

duplicate_count = econometric_data.duplicated(
    subset=["Date", "SubclassDescription"]
).sum()

missing_value_count = (
    econometric_data[required_columns]
    .isna()
    .sum()
    .sum()
)

subclass_month_counts = (
    econometric_data
    .groupby("SubclassDescription")["Date"]
    .nunique()
)

exchange_rate_columns = [
    "ExchangeRate",
    "Log_ExchangeRate",
    "ExchangeRate_Log_Change_Pct",
    "ExchangeRate_Positive_Cumulative_Pct",
    "ExchangeRate_Negative_Cumulative_Pct"
]

maximum_monthly_fx_values = (
    econometric_data
    .groupby("Date")[exchange_rate_columns]
    .nunique()
    .max()
    .max()
)

if duplicate_count != 0:
    raise ValueError("Duplicate subclass-month observations detected.")

if missing_value_count != 0:
    raise ValueError("Missing values detected in required variables.")

if subclass_month_counts.nunique() != 1:
    raise ValueError("Food subclasses do not have equal time coverage.")

if maximum_monthly_fx_values != 1:
    raise ValueError("Inconsistent exchange-rate values detected within months.")

validation_summary = pd.Series({
    "Observations": len(econometric_data),
    "Food subclasses": econometric_data[
        "SubclassDescription"
    ].nunique(),
    "Unique months": econometric_data["Date"].nunique(),
    "Minimum months per subclass": subclass_month_counts.min(),
    "Maximum months per subclass": subclass_month_counts.max(),
    "Duplicate subclass-month rows": duplicate_count,
    "Missing required values": missing_value_count,
    "Maximum FX values within a month": maximum_monthly_fx_values
})

validation_summary.to_frame(name="Value")

,Value
Observations,4830
Food subclasses,46
Unique months,105
Minimum months per subclass,105
Maximum months per subclass,105
Duplicate subclass-month rows,0
Missing required values,0
Maximum FX values within a month,1


### Modelling-Structure Validation

The validation confirms that the econometric dataset contains 4,830
observations representing 46 food subclasses across 105 common months.

Every subclass has complete coverage from April 2017 to December 2025. No duplicate subclass-month observations or missing modelling values were identified.

The exchange-rate variables are also consistent within each month, confirming that all food subclasses are matched to the same monthly macroeconomic series.

The dataset therefore satisfies the structural requirements for
subclass-specific time-series estimation.

### Econometric Modelling Framework

Two related econometric specifications will be estimated for each food
subclass.

### Symmetric ARDL Benchmark

The symmetric ARDL model uses log food CPI as the dependent variable and the log USD/ZAR exchange rate as the principal explanatory variable.

This specification assumes that Rand depreciation and appreciation have equal but opposite effects on food prices. It provides the conventional benchmark against which the asymmetric model can be assessed.

### Asymmetric NARDL Model

The NARDL specification replaces the single exchange-rate variable with its cumulative positive and negative components:

- `ExchangeRate_Positive_Cumulative_Pct` represents cumulative Rand
  depreciation.
- `ExchangeRate_Negative_Cumulative_Pct` represents cumulative Rand
  appreciation and remains negatively signed.

This decomposition allows depreciation and appreciation to have different short-run and long-run relationships with food prices.

### Dependent Variable

`Log_CPI` is used as the level-form dependent variable in the ARDL and NARDL specifications. Its first difference represents monthly food-price inflation.

Before estimating either model, the integration order of each variable must be assessed. ARDL bounds-testing methods permit a mixture of I(0) and I(1) variables but are not valid when any model variable is integrated of order two, I(2).

In [4]:
# stationarity-testing function
def run_stationarity_tests(
    series,
    variable_name,
    significance_level=0.05
):
    clean_series = (
        pd.Series(series)
        .dropna()
        .astype(float)
    )

    transformations = {
        "Level": clean_series,
        "First difference": clean_series.diff().dropna()
    }

    test_results = []

    for transformation, transformed_series in transformations.items():
        adf_result = adfuller(
            transformed_series,
            regression="c",
            autolag="AIC"
        )

        with warnings.catch_warnings():
            warnings.simplefilter(
                "ignore",
                category=InterpolationWarning
            )

            kpss_result = kpss(
                transformed_series,
                regression="c",
                nlags="auto"
            )

        test_results.append({
            "Variable": variable_name,
            "Transformation": transformation,
            "Observations": len(transformed_series),
            "ADF_Statistic": adf_result[0],
            "ADF_P_Value": adf_result[1],
            "ADF_Lags": adf_result[2],
            "ADF_Stationary": (
                adf_result[1] < significance_level
            ),
            "KPSS_Statistic": kpss_result[0],
            "KPSS_P_Value": kpss_result[1],
            "KPSS_Lags": kpss_result[2],
            "KPSS_Stationary": (
                kpss_result[1] >= significance_level
            )
        })

    return pd.DataFrame(test_results)

In [5]:
# prepare the unique monthly exchange-rate series
monthly_econometric_data = (
    econometric_data[
        [
            "Date",
            "Log_ExchangeRate",
            "ExchangeRate_Log_Change_Pct",
            "ExchangeRate_Positive_Cumulative_Pct",
            "ExchangeRate_Negative_Cumulative_Pct"
        ]
    ]
    .drop_duplicates(subset="Date")
    .sort_values("Date")
    .reset_index(drop=True)
)

exchange_rate_test_variables = [
    "Log_ExchangeRate",
    "ExchangeRate_Log_Change_Pct",
    "ExchangeRate_Positive_Cumulative_Pct",
    "ExchangeRate_Negative_Cumulative_Pct"
]

exchange_rate_stationarity_results = pd.concat(
    [
        run_stationarity_tests(
            monthly_econometric_data[variable],
            variable
        )
        for variable in exchange_rate_test_variables
    ],
    ignore_index=True
)

exchange_rate_stationarity_results

,Variable,Transformation,Observations,ADF_Statistic,ADF_P_Value,ADF_Lags,ADF_Stationary,KPSS_Statistic,KPSS_P_Value,KPSS_Lags,KPSS_Stationary
0,Log_ExchangeRate,Level,105,-1.864525,0.348929,1,False,1.280760,0.010000,6,False
1,Log_ExchangeRate,First difference,104,-8.134200,0.000000,0,True,0.071541,0.100000,1,True
2,ExchangeRate_Log_Change_Pct,Level,105,-8.288868,0.000000,0,True,0.080543,0.100000,1,True
3,ExchangeRate_Log_Change_Pct,First difference,104,-6.269094,0.000000,9,True,0.157573,0.100000,30,True
4,ExchangeRate_Positive_Cumulative_Pct,Level,105,-1.683814,0.439471,1,False,1.578454,0.010000,6,False
5,ExchangeRate_Positive_Cumulative_Pct,First difference,104,-8.252650,0.000000,0,True,0.293281,0.100000,2,True
6,ExchangeRate_Negative_Cumulative_Pct,Level,105,-1.505169,0.530953,0,False,1.586885,0.010000,6,False
7,ExchangeRate_Negative_Cumulative_Pct,First difference,104,-7.863987,0.000000,0,True,0.199321,0.100000,1,True


In [6]:
# test log CPI separately for each food subclass
food_price_stationarity_results = []

for subclass_name, subclass_data in econometric_data.groupby(
    "SubclassDescription"
):
    subclass_results = run_stationarity_tests(
        subclass_data["Log_CPI"],
        "Log_CPI"
    )

    subclass_results.insert(
        0,
        "SubclassDescription",
        subclass_name
    )

    food_price_stationarity_results.append(subclass_results)

food_price_stationarity_results = pd.concat(
    food_price_stationarity_results,
    ignore_index=True
)

food_price_stationarity_results.head(10)

,SubclassDescription,Variable,Transformation,Observations,ADF_Statistic,ADF_P_Value,ADF_Lags,ADF_Stationary,KPSS_Statistic,KPSS_P_Value,KPSS_Lags,KPSS_Stationary
0,Baby food,Log_CPI,Level,105,-1.216722,0.666361,7,False,1.486630,0.010000,6,False
1,Baby food,Log_CPI,First difference,104,-1.751753,0.404664,6,False,0.370346,0.089937,2,True
2,Bread and bakery products,Log_CPI,Level,105,-0.203780,0.938070,1,False,1.565412,0.010000,6,False
3,Bread and bakery products,Log_CPI,First difference,104,-6.827378,0.000000,0,True,0.149196,0.100000,4,True
4,Breakfast cereals,Log_CPI,Level,105,-0.678768,0.852173,0,False,1.551419,0.010000,6,False
5,Breakfast cereals,Log_CPI,First difference,104,-10.709926,0.000000,0,True,0.180325,0.100000,1,True
6,Cereals,Log_CPI,Level,105,-1.183943,0.680465,5,False,1.403764,0.010000,6,False
7,Cereals,Log_CPI,First difference,104,-4.254948,0.000531,2,True,0.202674,0.100000,2,True
8,Cheese,Log_CPI,Level,105,0.557774,0.986527,0,False,1.563846,0.010000,6,False
9,Cheese,Log_CPI,First difference,104,-10.890011,0.000000,0,True,0.209836,0.100000,0,True


In [7]:
# summarise stationarity decisions across food subclasses
food_price_stationarity_summary = (
    food_price_stationarity_results
    .groupby("Transformation")
    .agg(
        Food_Subclasses=(
            "SubclassDescription",
            "nunique"
        ),
        ADF_Stationary=(
            "ADF_Stationary",
            "sum"
        ),
        KPSS_Stationary=(
            "KPSS_Stationary",
            "sum"
        )
    )
    .reindex(["Level", "First difference"])
)

food_price_stationarity_summary

,Food_Subclasses,ADF_Stationary,KPSS_Stationary
Transformation,,,
Level,46,0,1
First difference,46,42,44


### Initial Stationarity Interpretation

The log exchange rate is non-stationary in levels but stationary after first differencing according to both the ADF and KPSS tests. It is therefore classified as I(1).

The cumulative depreciation and appreciation components follow the same
pattern. Both are non-stationary in levels and stationary after first
differencing, supporting their inclusion as I(1) variables in the NARDL
framework.

The monthly log exchange-rate change is stationary in levels and is therefore classified as I(0).

For food prices, none of the 46 subclass log CPI series was stationary in levels according to the ADF test, while only one was classified as stationary by the KPSS test. After first differencing, the ADF test classified 42 subclasses as stationary and the KPSS test classified 44 as stationary.

Most food-price series therefore appear to be I(1). However, subclasses for which the tests disagree or fail to establish first-difference stationarity must be examined before model estimation.

In [8]:
# classify food-price integration orders
level_test_results = (
    food_price_stationarity_results[
        food_price_stationarity_results[
            "Transformation"
        ] == "Level"
    ]
    .set_index("SubclassDescription")
    [
        [
            "ADF_P_Value",
            "ADF_Stationary",
            "KPSS_P_Value",
            "KPSS_Stationary"
        ]
    ]
    .rename(
        columns={
            "ADF_P_Value": "Level_ADF_P_Value",
            "ADF_Stationary": "Level_ADF_Stationary",
            "KPSS_P_Value": "Level_KPSS_P_Value",
            "KPSS_Stationary": "Level_KPSS_Stationary"
        }
    )
)

difference_test_results = (
    food_price_stationarity_results[
        food_price_stationarity_results[
            "Transformation"
        ] == "First difference"
    ]
    .set_index("SubclassDescription")
    [
        [
            "ADF_P_Value",
            "ADF_Stationary",
            "KPSS_P_Value",
            "KPSS_Stationary"
        ]
    ]
    .rename(
        columns={
            "ADF_P_Value": "Difference_ADF_P_Value",
            "ADF_Stationary": "Difference_ADF_Stationary",
            "KPSS_P_Value": "Difference_KPSS_P_Value",
            "KPSS_Stationary": "Difference_KPSS_Stationary"
        }
    )
)

food_price_integration_status = level_test_results.join(
    difference_test_results
)

level_stationary = (
    food_price_integration_status["Level_ADF_Stationary"]
    & food_price_integration_status["Level_KPSS_Stationary"]
)

difference_stationary = (
    food_price_integration_status["Difference_ADF_Stationary"]
    & food_price_integration_status["Difference_KPSS_Stationary"]
)

food_price_integration_status["Integration_Order"] = np.select(
    [
        level_stationary,
        ~level_stationary & difference_stationary
    ],
    [
        "I(0)",
        "I(1)"
    ],
    default="Requires review"
)

food_price_integration_status[
    "Integration_Order"
].value_counts().to_frame(name="Food_Subclasses")

,Food_Subclasses
Integration_Order,
I(1),40
Requires review,6


### Review of Inconclusive Food-Price Series

A conservative classification requires both tests to support stationarity.

Subclasses are classified as I(0) when both tests indicate stationarity in levels and as I(1) when both tests indicate stationarity after first differencing.

A `Requires review` result does not automatically mean that the series is I(2). It indicates that the two tests disagree or that first-difference stationarity has not yet been established conclusively.

In [9]:
# inspect subclasses with inconclusive results
food_price_series_for_review = (
    food_price_integration_status[
        food_price_integration_status[
            "Integration_Order"
        ] == "Requires review"
    ]
    .sort_values(
        [
            "Difference_ADF_P_Value",
            "Difference_KPSS_P_Value"
        ],
        ascending=False
    )
)

food_price_series_for_review

,Level_ADF_P_Value,Level_ADF_Stationary,Level_KPSS_P_Value,Level_KPSS_Stationary,Difference_ADF_P_Value,Difference_ADF_Stationary,Difference_KPSS_P_Value,Difference_KPSS_Stationary,Integration_Order
SubclassDescription,,,,,,,,,
"Stone fruits and pome fruits, fresh",0.943496,False,0.010000,False,0.525855,False,0.100000,True,Requires review
Baby food,0.666361,False,0.010000,False,0.404664,False,0.089937,True,Requires review
"Salt, condiments and sauces",0.905076,False,0.010000,False,0.348858,False,0.100000,True,Requires review
Other food products n.e.c.,0.858553,False,0.010000,False,0.152017,False,0.100000,True,Requires review
Coffee and coffee substitutes,0.998312,False,0.010000,False,0.001276,True,0.010000,False,Requires review
"Chocolate, cocoa, and cocoa-based food products",0.998883,False,0.010000,False,0.000000,True,0.010000,False,Requires review


### Robustness Tests for Inconclusive Series

Six food subclasses could not be classified using the initial joint decision rule.

For four subclasses, the KPSS test supports first-difference stationarity but the ADF test does not reject a unit root. For coffee and chocolate products, the ADF test supports stationarity but the KPSS test does not.

These series are reassessed using alternative ADF lag-selection rules and different KPSS bandwidths. This determines whether the conclusions are sensitive to a particular test specification.

The robustness checks are applied only to the six unresolved subclasses. They are not used to select whichever individual result is most favourable.

In [10]:
# test ADF sensitivity to lag selection
review_subclasses = (
    food_price_series_for_review
    .index
    .tolist()
)

adf_specifications = {
    "AIC": {
        "maxlag": 12,
        "autolag": "AIC"
    },
    "BIC": {
        "maxlag": 12,
        "autolag": "BIC"
    },
    "Fixed lag 0": {
        "maxlag": 0,
        "autolag": None
    },
    "Fixed lag 1": {
        "maxlag": 1,
        "autolag": None
    },
    "Fixed lag 3": {
        "maxlag": 3,
        "autolag": None
    },
    "Fixed lag 6": {
        "maxlag": 6,
        "autolag": None
    }
}

adf_robustness_records = []

for subclass_name in review_subclasses:
    subclass_series = (
        econometric_data[
            econometric_data["SubclassDescription"] == subclass_name
        ]
        .sort_values("Date")["Log_CPI"]
        .diff()
        .dropna()
    )

    for specification, settings in adf_specifications.items():
        test_result = adfuller(
            subclass_series,
            regression="c",
            **settings
        )

        adf_robustness_records.append({
            "SubclassDescription": subclass_name,
            "Specification": specification,
            "ADF_Statistic": test_result[0],
            "ADF_P_Value": test_result[1],
            "Selected_Lags": test_result[2],
            "Stationary": test_result[1] < 0.05
        })

adf_robustness_results = pd.DataFrame(
    adf_robustness_records
)

In [11]:
# display ADF p-values across specifications
adf_robustness_p_values = (
    adf_robustness_results
    .pivot(
        index="SubclassDescription",
        columns="Specification",
        values="ADF_P_Value"
    )
    .reindex(columns=adf_specifications.keys())
)

adf_robustness_p_values

Specification,AIC,BIC,Fixed lag 0,Fixed lag 1,Fixed lag 3,Fixed lag 6
SubclassDescription,,,,,,
Baby food,0.404664,0.000000,0.000000,0.000000,0.000172,0.404664
"Chocolate, cocoa, and cocoa-based food products",0.339139,0.000000,0.000000,0.000000,0.000164,0.079142
Coffee and coffee substitutes,0.001276,0.000000,0.000000,0.000000,0.002514,0.093880
Other food products n.e.c.,0.242231,0.000000,0.000000,0.000000,0.000527,0.242231
"Salt, condiments and sauces",0.348858,0.000000,0.000000,0.000000,0.000012,0.009600
"Stone fruits and pome fruits, fresh",0.525855,0.309753,0.000000,0.000000,0.000003,0.000001


In [12]:
# display lags used in each ADF specification
adf_selected_lags = (
    adf_robustness_results
    .pivot(
        index="SubclassDescription",
        columns="Specification",
        values="Selected_Lags"
    )
    .reindex(columns=adf_specifications.keys())
)

adf_selected_lags

Specification,AIC,BIC,Fixed lag 0,Fixed lag 1,Fixed lag 3,Fixed lag 6
SubclassDescription,,,,,,
Baby food,6,0,0,1,3,6
"Chocolate, cocoa, and cocoa-based food products",8,0,0,1,3,6
Coffee and coffee substitutes,2,0,0,1,3,6
Other food products n.e.c.,6,0,0,1,3,6
"Salt, condiments and sauces",12,0,0,1,3,6
"Stone fruits and pome fruits, fresh",12,11,0,1,3,6


In [13]:
# test KPSS sensitivity to bandwidth selection
kpss_specifications = {
    "Automatic": "auto",
    "Lag 1": 1,
    "Lag 3": 3,
    "Lag 6": 6,
    "Lag 12": 12
}

kpss_robustness_records = []

for subclass_name in review_subclasses:
    subclass_series = (
        econometric_data[
            econometric_data["SubclassDescription"] == subclass_name
        ]
        .sort_values("Date")["Log_CPI"]
        .diff()
        .dropna()
    )

    for specification, lag_setting in kpss_specifications.items():
        with warnings.catch_warnings():
            warnings.simplefilter(
                "ignore",
                category=InterpolationWarning
            )

            test_result = kpss(
                subclass_series,
                regression="c",
                nlags=lag_setting
            )

        kpss_robustness_records.append({
            "SubclassDescription": subclass_name,
            "Specification": specification,
            "KPSS_Statistic": test_result[0],
            "KPSS_P_Value": test_result[1],
            "Selected_Lags": test_result[2],
            "Stationary": test_result[1] >= 0.05
        })

kpss_robustness_results = pd.DataFrame(
    kpss_robustness_records
)

In [14]:
# display KPSS p-values across bandwidths
kpss_robustness_p_values = (
    kpss_robustness_results
    .pivot(
        index="SubclassDescription",
        columns="Specification",
        values="KPSS_P_Value"
    )
    .reindex(columns=kpss_specifications.keys())
)

kpss_robustness_p_values

Specification,Automatic,Lag 1,Lag 3,Lag 6,Lag 12
SubclassDescription,,,,,
Baby food,0.089937,0.076703,0.094950,0.100000,0.100000
"Chocolate, cocoa, and cocoa-based food products",0.010000,0.010000,0.010000,0.010000,0.040656
Coffee and coffee substitutes,0.010000,0.010000,0.010000,0.010000,0.021132
Other food products n.e.c.,0.100000,0.100000,0.100000,0.100000,0.100000
"Salt, condiments and sauces",0.100000,0.100000,0.100000,0.100000,0.100000
"Stone fruits and pome fruits, fresh",0.100000,0.100000,0.100000,0.100000,0.100000


### Robustness-Test Interpretation

The robustness tests indicate that the initial ADF failures for baby food, other food products, salt, condiments and sauces, and stone fruits are sensitive to the number of lags included in the test.

For baby food and other food products, the AIC specification selected six lags and failed to reject a unit root. However, BIC selected zero lags, and the fixed zero-, one- and three-lag specifications strongly supported first-difference stationarity. The KPSS test also supported stationarity across all bandwidths.

Salt, condiments and sauces followed a similar pattern. AIC selected 12 lags and failed to reject a unit root, while BIC and the fixed-lag specifications supported stationarity.

Stone fruits and pome fruits remained sensitive under the high-lag ADF
specifications. However, the fixed zero-, one-, three- and six-lag tests rejected a unit root, while KPSS consistently supported stationarity. The series is therefore consistent with I(1), although its classification is lag-sensitive.

Coffee and chocolate products require additional investigation. Their ADF results generally support first-difference stationarity, but KPSS rejects stationarity across all tested bandwidths. A structural break may account for this disagreement.

### Structural-Break Stationarity Test

The Zivot-Andrews test is used for the two subclasses with persistent
ADF–KPSS disagreement.

Unlike the standard ADF test, the Zivot-Andrews test allows for one
endogenously identified structural break. Its null hypothesis is that the series contains a unit root, including when a possible structural break is considered.

A p-value below 0.05 supports stationarity around a structural break and provides evidence against the series being I(2).

In [15]:
# test first differences with one structural break
structural_break_subclasses = [
    "Coffee and coffee substitutes",
    "Chocolate, cocoa, and cocoa-based food products"
]

structural_break_records = []

for subclass_name in structural_break_subclasses:
    subclass_series = (
        econometric_data[
            econometric_data["SubclassDescription"] == subclass_name
        ]
        .sort_values("Date")
        .set_index("Date")["Log_CPI"]
        .diff()
        .dropna()
    )

    test_result = zivot_andrews(
        subclass_series,
        trim=0.15,
        maxlag=12,
        regression="c",
        autolag="BIC"
    )

    break_position = test_result[4]
    break_date = subclass_series.index[break_position]

    structural_break_records.append({
        "SubclassDescription": subclass_name,
        "ZA_Statistic": test_result[0],
        "ZA_P_Value": test_result[1],
        "Selected_Lags": test_result[3],
        "Break_Date": break_date,
        "Stationary_With_Break": test_result[1] < 0.05
    })

structural_break_results = pd.DataFrame(
    structural_break_records
).set_index("SubclassDescription")

structural_break_results

,ZA_Statistic,ZA_P_Value,Selected_Lags,Break_Date,Stationary_With_Break
SubclassDescription,,,,,
Coffee and coffee substitutes,-12.992023,0.000010,0,2022-01-01,True
"Chocolate, cocoa, and cocoa-based food products",-11.477148,0.000010,0,2024-03-01,True


### Structural-Break Test Interpretation

The Zivot-Andrews test strongly rejects the unit-root null for the first differences of both unresolved food-price series.

Coffee and coffee substitutes recorded a structural-break date of January 2022, while chocolate, cocoa and cocoa-based food products recorded a break in March 2024.

These results indicate that the first differences are stationary when a
possible structural break is considered. The earlier ADF–KPSS disagreement therefore does not provide evidence that either series is I(2).

Both subclasses are retained in the econometric sample and classified as I(1),with their structural-break sensitivity documented for subsequent model interpretation.

In [16]:
# finalise food-price integration classifications
lag_sensitive_subclasses = [
    "Baby food",
    "Other food products n.e.c.",
    "Salt, condiments and sauces",
    "Stone fruits and pome fruits, fresh"
]

break_adjusted_subclasses = [
    "Coffee and coffee substitutes",
    "Chocolate, cocoa, and cocoa-based food products"
]

food_price_integration_status["Classification_Basis"] = np.where(
    food_price_integration_status["Integration_Order"] == "I(1)",
    "ADF and KPSS agree",
    pd.NA
)

food_price_integration_status.loc[
    lag_sensitive_subclasses,
    ["Integration_Order", "Classification_Basis"]
] = [
    "I(1)",
    "Stationary after differencing; ADF lag-sensitive"
]

food_price_integration_status.loc[
    break_adjusted_subclasses,
    ["Integration_Order", "Classification_Basis"]
] = [
    "I(1)",
    "Stationary after differencing with structural break"
]

unresolved_series = food_price_integration_status[
    ~food_price_integration_status["Integration_Order"].isin(
        ["I(0)", "I(1)"]
    )
]

if not unresolved_series.empty:
    raise ValueError(
        "Some food-price series remain unresolved."
    )

food_price_integration_status[
    ["Integration_Order", "Classification_Basis"]
].head(10)

,Integration_Order,Classification_Basis
SubclassDescription,,
Baby food,I(1),Stationary after differencing; ADF lag-sensitive
Bread and bakery products,I(1),ADF and KPSS agree
Breakfast cereals,I(1),ADF and KPSS agree
Cereals,I(1),ADF and KPSS agree
Cheese,I(1),ADF and KPSS agree
"Chocolate, cocoa, and cocoa-based food products",I(1),Stationary after differencing with structural ...
Coffee and coffee substitutes,I(1),Stationary after differencing with structural ...
"Dates, figs and tropical fruits, fresh",I(1),ADF and KPSS agree
Eggs,I(1),ADF and KPSS agree


In [17]:
# summarise integration orders for all model components
model_integration_summary = pd.DataFrame({
    "Model_Component": [
        "Food-price level",
        "Symmetric exchange-rate level",
        "Monthly exchange-rate change",
        "Cumulative depreciation component",
        "Cumulative appreciation component"
    ],
    "Variable": [
        "Log_CPI",
        "Log_ExchangeRate",
        "ExchangeRate_Log_Change_Pct",
        "ExchangeRate_Positive_Cumulative_Pct",
        "ExchangeRate_Negative_Cumulative_Pct"
    ],
    "Integration_Order": [
        "I(1)",
        "I(1)",
        "I(0)",
        "I(1)",
        "I(1)"
    ],
    "Scope": [
        "46 subclass series",
        "Common monthly series",
        "Common monthly series",
        "Common monthly series",
        "Common monthly series"
    ]
})

model_integration_summary

,Model_Component,Variable,Integration_Order,Scope
0,Food-price level,Log_CPI,I(1),46 subclass series
1,Symmetric exchange-rate level,Log_ExchangeRate,I(1),Common monthly series
2,Monthly exchange-rate change,ExchangeRate_Log_Change_Pct,I(0),Common monthly series
3,Cumulative depreciation component,ExchangeRate_Positive_Cumulative_Pct,I(1),Common monthly series
4,Cumulative appreciation component,ExchangeRate_Negative_Cumulative_Pct,I(1),Common monthly series


### Stationarity Conclusion

All 46 log food CPI series are classified as I(1). Forty subclasses received consistent ADF and KPSS support, four were classified after examining lag-selection sensitivity, and two were classified using a
structural-break-aware test.

The log exchange rate and its cumulative positive and negative components are also I(1), while the monthly log exchange-rate change is I(0).

No variable included in the proposed econometric specifications was found to be I(2). The integration-order results therefore permit ARDL and NARDL estimation.

These findings establish model eligibility but do not by themselves establish cointegration. The existence of a long-run relationship must be assessed using the bounds-testing procedure after model estimation.

### Lag-Order Selection

ARDL model performance and interpretation depend on the number of historical food-price and exchange-rate observations included.

The lag-selection procedure considers autoregressive and distributed lag orders from one to six months. This range captures short- and medium-term monthly adjustment while remaining parsimonious for the available sample of 105 observations per subclass.

A minimum lag order of one is imposed so that every selected ARDL model can subsequently be represented as an unrestricted error-correction model for bounds testing.

The Bayesian Information Criterion is used to select the preferred model. BIC penalises unnecessary parameters more strongly than AIC and is therefore appropriate for limiting overfitting in the subclass-specific models.

All candidate models:

- include a constant
- include monthly seasonal indicators
- allow a contemporaneous exchange-rate relationship
- use the same six-month estimation hold-back
- are compared using the same effective sample period

In [18]:
# lag-selection settings
maximum_price_lag = 6
maximum_exchange_rate_lag = 6
selection_hold_back = 6
seasonal_period = 12

lag_selection_settings = pd.Series({
    "Maximum food-price lag": maximum_price_lag,
    "Maximum exchange-rate lag": maximum_exchange_rate_lag,
    "Minimum permitted lag": 1,
    "Selection criterion": "BIC",
    "Seasonal indicators": True,
    "Seasonal period": seasonal_period,
    "Common hold-back": selection_hold_back
})

lag_selection_settings.to_frame(name="Setting")

,Setting
Maximum food-price lag,6
Maximum exchange-rate lag,6
Minimum permitted lag,1
Selection criterion,BIC
Seasonal indicators,True
Seasonal period,12
Common hold-back,6


In [19]:
# select the lowest-BIC ARDL specification
def select_best_ardl_model(
    subclass_data,
    exogenous_variables
):
    model_data = (
        subclass_data
        .sort_values("Date")
        .set_index("Date")
        .asfreq("MS")
    )

    best_result = None
    best_bic = np.inf
    best_price_lag = None
    best_exchange_rate_lag = None
    successful_candidates = 0

    for price_lag in range(1, maximum_price_lag + 1):
        for exchange_rate_lag in range(
            1,
            maximum_exchange_rate_lag + 1
        ):
            try:
                candidate_model = ARDL(
                    endog=model_data["Log_CPI"],
                    lags=price_lag,
                    exog=model_data[exogenous_variables],
                    order=exchange_rate_lag,
                    trend="c",
                    seasonal=True,
                    period=seasonal_period,
                    causal=False,
                    hold_back=selection_hold_back,
                    missing="raise"
                )

                candidate_result = candidate_model.fit()
                successful_candidates += 1

            except (ValueError, np.linalg.LinAlgError):
                continue

            if candidate_result.bic < best_bic:
                best_result = candidate_result
                best_bic = candidate_result.bic
                best_price_lag = price_lag
                best_exchange_rate_lag = exchange_rate_lag

    if best_result is None:
        subclass_name = subclass_data[
            "SubclassDescription"
        ].iloc[0]

        raise ValueError(
            f"No valid ARDL specification found for {subclass_name}."
        )

    selection_summary = {
        "Price_Lag": best_price_lag,
        "Exchange_Rate_Lag": best_exchange_rate_lag,
        "BIC": best_bic,
        "Observations": int(best_result.nobs),
        "Parameters": len(best_result.params),
        "Successful_Candidates": successful_candidates
    }

    return selection_summary, best_result

### Symmetric ARDL Lag Selection

The symmetric specification uses `Log_CPI` as the dependent variable and `Log_ExchangeRate` as the explanatory variable.

For each food subclass, 36 candidate specifications are evaluated:

1 ≤ p ≤ 6 and 1 ≤ q ≤ 6, where p,q∈Z

where p is the food-price lag order and q is the exchange-rate lag
order.

The specification with the lowest BIC is retained for that subclass.

In [20]:
# select the symmetric ARDL order for each subclass
symmetric_selection_records = []
symmetric_ardl_results = {}

for subclass_name, subclass_data in econometric_data.groupby(
    "SubclassDescription"
):
    selection_summary, fitted_result = select_best_ardl_model(
        subclass_data=subclass_data,
        exogenous_variables=["Log_ExchangeRate"]
    )

    selection_summary["SubclassDescription"] = subclass_name
    symmetric_selection_records.append(selection_summary)

    symmetric_ardl_results[subclass_name] = fitted_result

symmetric_lag_selection = (
    pd.DataFrame(symmetric_selection_records)
    [
        [
            "SubclassDescription",
            "Price_Lag",
            "Exchange_Rate_Lag",
            "BIC",
            "Observations",
            "Parameters",
            "Successful_Candidates"
        ]
    ]
    .sort_values("SubclassDescription")
    .reset_index(drop=True)
)

print(
    "Symmetric models selected:",
    len(symmetric_ardl_results)
)

symmetric_lag_selection.head(10)

Symmetric models selected: 46


,SubclassDescription,Price_Lag,Exchange_Rate_Lag,BIC,Observations,Parameters,Successful_Candidates
0,Baby food,2,1,-559.506177,99,16,36
1,Bread and bakery products,3,1,-643.806929,99,17,36
2,Breakfast cereals,1,1,-517.008894,99,15,36
3,Cereals,1,1,-499.488882,99,15,36
4,Cheese,2,1,-579.678465,99,16,36
5,"Chocolate, cocoa, and cocoa-based food products",1,1,-606.623386,99,15,36
6,Coffee and coffee substitutes,1,1,-494.256402,99,15,36
7,"Dates, figs and tropical fruits, fresh",1,1,-301.613661,99,15,36
8,Eggs,2,1,-453.100335,99,16,36
9,Fish,1,1,-657.432487,99,15,36


In [21]:
# summarise the selected symmetric lag orders
symmetric_lag_distribution = (
    symmetric_lag_selection
    .groupby(
        [
            "Price_Lag",
            "Exchange_Rate_Lag"
        ]
    )
    .size()
    .rename("Food_Subclasses")
    .reset_index()
    .sort_values(
        [
            "Food_Subclasses",
            "Price_Lag",
            "Exchange_Rate_Lag"
        ],
        ascending=[False, True, True]
    )
    .reset_index(drop=True)
)

print(
    "Models with all 36 candidates estimated:",
    (
        symmetric_lag_selection[
            "Successful_Candidates"
        ] == 36
    ).sum()
)

print(
    "Selected price-lag range:",
    symmetric_lag_selection["Price_Lag"].min(),
    "to",
    symmetric_lag_selection["Price_Lag"].max()
)

print(
    "Selected exchange-rate-lag range:",
    symmetric_lag_selection["Exchange_Rate_Lag"].min(),
    "to",
    symmetric_lag_selection["Exchange_Rate_Lag"].max()
)

symmetric_lag_distribution

Models with all 36 candidates estimated: 46
Selected price-lag range: 1 to 3
Selected exchange-rate-lag range: 1 to 5


,Price_Lag,Exchange_Rate_Lag,Food_Subclasses
0,1,1,25
1,2,1,13
2,3,1,4
3,1,2,1
4,2,2,1
5,3,3,1
6,3,5,1


### Symmetric Lag-Selection Results

All 46 food subclasses successfully estimated the complete set of 36
candidate symmetric ARDL models. The common six-month hold-back resulted in 99 estimation observations for every candidate.

The most common specification was ARDL(1, 1), selected for 25 subclasses. ARDL(2, 1) was selected for 13 subclasses, while ARDL(3, 1) was selected for four subclasses.

Overall, 42 of the 46 subclasses selected an exchange-rate lag order of one. Only four subclasses selected longer exchange-rate lag structures, with the maximum selected order being five.

The results suggest that food-price persistence generally requires between one and three months of autoregressive information, while the symmetric exchange-rate contribution is usually concentrated in the current and previous month.

These lag selections describe the preferred dynamic structure. They do not yet establish the existence or magnitude of exchange-rate pass-through.

In [22]:
# select the asymmetric NARDL order for each subclass
asymmetric_selection_records = []
nardl_ardl_results = {}

asymmetric_variables = [
    "ExchangeRate_Positive_Cumulative_Pct",
    "ExchangeRate_Negative_Cumulative_Pct"
]

for subclass_name, subclass_data in econometric_data.groupby(
    "SubclassDescription"
):
    selection_summary, fitted_result = select_best_ardl_model(
        subclass_data=subclass_data,
        exogenous_variables=asymmetric_variables
    )

    selection_summary["SubclassDescription"] = subclass_name
    selection_summary["Component_Lag"] = selection_summary.pop(
        "Exchange_Rate_Lag"
    )

    asymmetric_selection_records.append(selection_summary)
    nardl_ardl_results[subclass_name] = fitted_result

asymmetric_lag_selection = (
    pd.DataFrame(asymmetric_selection_records)
    [
        [
            "SubclassDescription",
            "Price_Lag",
            "Component_Lag",
            "BIC",
            "Observations",
            "Parameters",
            "Successful_Candidates"
        ]
    ]
    .sort_values("SubclassDescription")
    .reset_index(drop=True)
)

print(
    "Asymmetric models selected:",
    len(nardl_ardl_results)
)

asymmetric_lag_selection.head(10)

Asymmetric models selected: 46


,SubclassDescription,Price_Lag,Component_Lag,BIC,Observations,Parameters,Successful_Candidates
0,Baby food,2,1,-555.905597,99,18,36
1,Bread and bakery products,3,1,-644.536587,99,19,36
2,Breakfast cereals,1,1,-512.221850,99,17,36
3,Cereals,1,2,-495.847916,99,19,36
4,Cheese,2,1,-577.142052,99,18,36
5,"Chocolate, cocoa, and cocoa-based food products",1,1,-600.988656,99,17,36
6,Coffee and coffee substitutes,1,1,-489.995329,99,17,36
7,"Dates, figs and tropical fruits, fresh",2,1,-300.584765,99,18,36
8,Eggs,2,1,-444.354010,99,18,36
9,Fish,1,1,-654.240195,99,17,36


In [23]:
# summarise the selected asymmetric lag orders
asymmetric_lag_distribution = (
    asymmetric_lag_selection
    .groupby(
        [
            "Price_Lag",
            "Component_Lag"
        ]
    )
    .size()
    .rename("Food_Subclasses")
    .reset_index()
    .sort_values(
        [
            "Food_Subclasses",
            "Price_Lag",
            "Component_Lag"
        ],
        ascending=[False, True, True]
    )
    .reset_index(drop=True)
)

print(
    "Models with all 36 candidates estimated:",
    (
        asymmetric_lag_selection[
            "Successful_Candidates"
        ] == 36
    ).sum()
)

print(
    "Selected price-lag range:",
    asymmetric_lag_selection["Price_Lag"].min(),
    "to",
    asymmetric_lag_selection["Price_Lag"].max()
)

print(
    "Selected component-lag range:",
    asymmetric_lag_selection["Component_Lag"].min(),
    "to",
    asymmetric_lag_selection["Component_Lag"].max()
)

asymmetric_lag_distribution

Models with all 36 candidates estimated: 46
Selected price-lag range: 1 to 4
Selected component-lag range: 1 to 2


,Price_Lag,Component_Lag,Food_Subclasses
0,1,1,22
1,2,1,18
2,1,2,2
3,3,1,2
4,3,2,1
5,4,2,1


### Asymmetric Lag-Selection Results

All 46 food subclasses successfully estimated the complete set of 36
candidate asymmetric models.

The most common specification was NARDL(1, 1, 1), selected for 22 subclasses. NARDL(2, 1, 1) was selected for 18 subclasses. Together, these two specifications account for 40 of the 46 food subclasses.

The selected food-price lag orders range from one to four months. The
depreciation and appreciation components require no more than two lags, with 42 subclasses selecting a common component lag of one.

The selected models remain parsimonious despite allowing depreciation and appreciation to have separate coefficients. Formal bounds and asymmetry tests are still required before interpreting these relationships as evidence of asymmetric exchange-rate pass-through.

In [24]:
# compare selected symmetric and asymmetric models
symmetric_bic_results = (
    symmetric_lag_selection[
        [
            "SubclassDescription",
            "Price_Lag",
            "Exchange_Rate_Lag",
            "BIC",
            "Parameters"
        ]
    ]
    .rename(
        columns={
            "Price_Lag": "ARDL_Price_Lag",
            "Exchange_Rate_Lag": "ARDL_Exchange_Rate_Lag",
            "BIC": "ARDL_BIC",
            "Parameters": "ARDL_Parameters"
        }
    )
)

asymmetric_bic_results = (
    asymmetric_lag_selection[
        [
            "SubclassDescription",
            "Price_Lag",
            "Component_Lag",
            "BIC",
            "Parameters"
        ]
    ]
    .rename(
        columns={
            "Price_Lag": "NARDL_Price_Lag",
            "Component_Lag": "NARDL_Component_Lag",
            "BIC": "NARDL_BIC",
            "Parameters": "NARDL_Parameters"
        }
    )
)

bic_model_comparison = symmetric_bic_results.merge(
    asymmetric_bic_results,
    on="SubclassDescription",
    how="inner",
    validate="one_to_one"
)

bic_model_comparison["BIC_Difference"] = (
    bic_model_comparison["NARDL_BIC"]
    - bic_model_comparison["ARDL_BIC"]
)

bic_model_comparison["Lower_BIC_Model"] = np.where(
    bic_model_comparison["BIC_Difference"] < 0,
    "NARDL",
    "ARDL"
)

bic_model_comparison.head(10)

,SubclassDescription,ARDL_Price_Lag,ARDL_Exchange_Rate_Lag,ARDL_BIC,ARDL_Parameters,NARDL_Price_Lag,NARDL_Component_Lag,NARDL_BIC,NARDL_Parameters,BIC_Difference,Lower_BIC_Model
0,Baby food,2,1,-559.506177,16,2,1,-555.905597,18,3.600580,ARDL
1,Bread and bakery products,3,1,-643.806929,17,3,1,-644.536587,19,-0.729657,NARDL
2,Breakfast cereals,1,1,-517.008894,15,1,1,-512.221850,17,4.787044,ARDL
3,Cereals,1,1,-499.488882,15,1,2,-495.847916,19,3.640966,ARDL
4,Cheese,2,1,-579.678465,16,2,1,-577.142052,18,2.536413,ARDL
5,"Chocolate, cocoa, and cocoa-based food products",1,1,-606.623386,15,1,1,-600.988656,17,5.634730,ARDL
6,Coffee and coffee substitutes,1,1,-494.256402,15,1,1,-489.995329,17,4.261073,ARDL
7,"Dates, figs and tropical fruits, fresh",1,1,-301.613661,15,2,1,-300.584765,18,1.028896,ARDL
8,Eggs,2,1,-453.100335,16,2,1,-444.354010,18,8.746326,ARDL
9,Fish,1,1,-657.432487,15,1,1,-654.240195,17,3.192293,ARDL


In [25]:
# summarise the BIC comparison
bic_comparison_summary = pd.Series({
    "Food subclasses compared": len(bic_model_comparison),
    "ARDL lower BIC": (
        bic_model_comparison["Lower_BIC_Model"] == "ARDL"
    ).sum(),
    "NARDL lower BIC": (
        bic_model_comparison["Lower_BIC_Model"] == "NARDL"
    ).sum(),
    "Mean BIC difference": (
        bic_model_comparison["BIC_Difference"].mean()
    ),
    "Median BIC difference": (
        bic_model_comparison["BIC_Difference"].median()
    ),
    "Minimum BIC difference": (
        bic_model_comparison["BIC_Difference"].min()
    ),
    "Maximum BIC difference": (
        bic_model_comparison["BIC_Difference"].max()
    )
})

bic_comparison_summary.to_frame(name="Value")

,Value
Food subclasses compared,46.000000
ARDL lower BIC,41.000000
NARDL lower BIC,5.000000
Mean BIC difference,3.548831
Median BIC difference,3.810502
Minimum BIC difference,-2.832619
Maximum BIC difference,8.746326


In [26]:
# display subclasses most favourable to NARDL
bic_model_comparison[
    [
        "SubclassDescription",
        "ARDL_BIC",
        "NARDL_BIC",
        "BIC_Difference",
        "Lower_BIC_Model"
    ]
].sort_values("BIC_Difference").head(10)

,SubclassDescription,ARDL_BIC,NARDL_BIC,BIC_Difference,Lower_BIC_Model
14,"Fruit-bearing vegetables, fresh or chilled",-316.324487,-319.157106,-2.832619,NARDL
30,"Other sugar, confectionery and dessert",-533.586424,-534.853262,-1.266838,NARDL
16,"Leafy or stem vegetables, fresh or chilled",-436.516521,-437.510754,-0.994233,NARDL
1,Bread and bakery products,-643.806929,-644.536587,-0.729657,NARDL
18,Margarine and similar preparations,-517.477554,-517.768269,-0.290716,NARDL
45,Yoghurt and similar products,-590.040817,-590.008786,0.032031,ARDL
42,Vegetables and pulses ground and other prepara...,-660.977909,-660.893018,0.084891,ARDL
41,Vegetable oils,-420.876325,-420.037287,0.839038,ARDL
20,"Meat, fresh, chilled or frozen",-632.258659,-631.330958,0.927702,ARDL
7,"Dates, figs and tropical fruits, fresh",-301.613661,-300.584765,1.028896,ARDL


### BIC Comparison Results

The symmetric ARDL specification recorded a lower BIC for 41 of the 46 food subclasses. NARDL recorded a lower BIC for five subclasses.

The mean BIC difference was 3.55 and the median difference was 3.81, both in favour of the symmetric specification.

The largest improvement from using NARDL occurred for fresh or chilled
fruit-bearing vegetables, where the NARDL BIC was 2.83 points lower than the symmetric ARDL BIC. The remaining NARDL improvements were smaller.

These results indicate that separating depreciation and appreciation does not generally improve penalised in-sample fit across all food subclasses. However, BIC is a model-selection measure rather than a formal test of cointegration or asymmetry. Both specifications are therefore retained for subsequent bounds and coefficient-restriction testing.

In [27]:
# estimate UECM models and perform bounds tests
def run_bounds_tests(
    fitted_ardl_models,
    model_name
):
    bounds_records = []
    fitted_uecm_models = {}

    for subclass_name, ardl_result in fitted_ardl_models.items():
        uecm_model = UECM.from_ardl(
            ardl_result.model
        )

        uecm_result = uecm_model.fit()
        bounds_result = uecm_result.bounds_test(
            case=3
        )

        critical_values = bounds_result.crit_vals
        lower_bound = critical_values.loc[95.0, "lower"]
        upper_bound = critical_values.loc[95.0, "upper"]
        test_statistic = bounds_result.stat

        if test_statistic > upper_bound:
            decision = "Cointegration"
        elif test_statistic < lower_bound:
            decision = "No cointegration"
        else:
            decision = "Inconclusive"

        bounds_records.append({
            "SubclassDescription": subclass_name,
            "Model": model_name,
            "Bounds_Statistic": test_statistic,
            "Lower_Bound_5pct": lower_bound,
            "Upper_Bound_5pct": upper_bound,
            "Lower_P_Value": bounds_result.p_values["lower"],
            "Upper_P_Value": bounds_result.p_values["upper"],
            "Decision": decision,
            "Observations": int(uecm_result.nobs)
        })

        fitted_uecm_models[subclass_name] = uecm_result

    bounds_table = (
        pd.DataFrame(bounds_records)
        .sort_values("SubclassDescription")
        .reset_index(drop=True)
    )

    return bounds_table, fitted_uecm_models

In [28]:
# test symmetric ARDL level relationships
symmetric_bounds_results, symmetric_uecm_results = (
    run_bounds_tests(
        fitted_ardl_models=symmetric_ardl_results,
        model_name="ARDL"
    )
)

print(
    "Symmetric bounds tests completed:",
    len(symmetric_bounds_results)
)

symmetric_bounds_results.head(10)

Symmetric bounds tests completed: 46


,SubclassDescription,Model,Bounds_Statistic,Lower_Bound_5pct,Upper_Bound_5pct,Lower_P_Value,Upper_P_Value,Decision,Observations
0,Baby food,ARDL,0.940153,3.802263,4.811686,0.721101,0.873765,No cointegration,99
1,Bread and bakery products,ARDL,0.568116,3.802263,4.811686,0.864023,0.950236,No cointegration,99
2,Breakfast cereals,ARDL,0.374619,3.802263,4.811686,0.927966,0.976344,No cointegration,99
3,Cereals,ARDL,5.253907,3.802263,4.811686,0.008950,0.031343,Cointegration,99
4,Cheese,ARDL,2.081091,3.802263,4.811686,0.300977,0.518048,No cointegration,99
5,"Chocolate, cocoa, and cocoa-based food products",ARDL,7.502411,3.802263,4.811686,0.000585,0.002640,Cointegration,99
6,Coffee and coffee substitutes,ARDL,4.413718,3.802263,4.811686,0.024301,0.073969,Inconclusive,99
7,"Dates, figs and tropical fruits, fresh",ARDL,4.380068,3.802263,4.811686,0.025279,0.076468,Inconclusive,99
8,Eggs,ARDL,3.284519,3.802263,4.811686,0.088137,0.210997,No cointegration,99
9,Fish,ARDL,0.241615,3.802263,4.811686,0.963387,0.988611,No cointegration,99


In [29]:
# test asymmetric NARDL level relationships
asymmetric_bounds_results, nardl_uecm_results = (
    run_bounds_tests(
        fitted_ardl_models=nardl_ardl_results,
        model_name="NARDL"
    )
)

print(
    "Asymmetric bounds tests completed:",
    len(asymmetric_bounds_results)
)

asymmetric_bounds_results.head(10)

Asymmetric bounds tests completed: 46


,SubclassDescription,Model,Bounds_Statistic,Lower_Bound_5pct,Upper_Bound_5pct,Lower_P_Value,Upper_P_Value,Decision,Observations
0,Baby food,NARDL,2.427553,3.229022,4.322282,0.149247,0.383795,No cointegration,99
1,Bread and bakery products,NARDL,2.509330,3.229022,4.322282,0.134225,0.357847,No cointegration,99
2,Breakfast cereals,NARDL,1.922801,3.229022,4.322282,0.276159,0.567264,No cointegration,99
3,Cereals,NARDL,6.620570,3.229022,4.322282,0.000259,0.002165,Cointegration,99
4,Cheese,NARDL,2.660325,3.229022,4.322282,0.109918,0.312780,No cointegration,99
5,"Chocolate, cocoa, and cocoa-based food products",NARDL,6.235160,3.229022,4.322282,0.000479,0.003765,Cointegration,99
6,Coffee and coffee substitutes,NARDL,4.032052,3.229022,4.322282,0.015188,0.071391,Inconclusive,99
7,"Dates, figs and tropical fruits, fresh",NARDL,9.173770,3.229022,4.322282,0.000004,0.000049,Cointegration,99
8,Eggs,NARDL,2.445594,3.229022,4.322282,0.145814,0.377982,No cointegration,99
9,Fish,NARDL,2.743821,3.229022,4.322282,0.098222,0.289517,No cointegration,99


In [30]:
# summarise the bounds-test decisions
combined_bounds_results = pd.concat(
    [
        symmetric_bounds_results,
        asymmetric_bounds_results
    ],
    ignore_index=True
)

bounds_decision_summary = (
    combined_bounds_results
    .groupby(
        [
            "Model",
            "Decision"
        ]
    )
    .size()
    .rename("Food_Subclasses")
    .reset_index()
)

bounds_decision_summary

,Model,Decision,Food_Subclasses
0,ARDL,Cointegration,6
1,ARDL,Inconclusive,4
2,ARDL,No cointegration,36
3,NARDL,Cointegration,9
4,NARDL,Inconclusive,6
5,NARDL,No cointegration,31


In [31]:
# inspect inconclusive bounds-test results
inconclusive_bounds_results = (
    combined_bounds_results[
        combined_bounds_results["Decision"] == "Inconclusive"
    ]
    [
        [
            "SubclassDescription",
            "Model",
            "Bounds_Statistic",
            "Lower_Bound_5pct",
            "Upper_Bound_5pct",
            "Lower_P_Value",
            "Upper_P_Value"
        ]
    ]
    .sort_values(
        [
            "Model",
            "Bounds_Statistic"
        ],
        ascending=[True, False]
    )
)

inconclusive_bounds_results

,SubclassDescription,Model,Bounds_Statistic,Lower_Bound_5pct,Upper_Bound_5pct,Lower_P_Value,Upper_P_Value
32,Pulses,ARDL,4.786498,3.802263,4.811686,0.015645,0.050864
6,Coffee and coffee substitutes,ARDL,4.413718,3.802263,4.811686,0.024301,0.073969
7,"Dates, figs and tropical fruits, fresh",ARDL,4.380068,3.802263,4.811686,0.025279,0.076468
27,"Other fruits, fresh",ARDL,3.881242,3.802263,4.811686,0.045082,0.123518
68,Milk,NARDL,4.057681,3.229022,4.322282,0.014608,0.069208
52,Coffee and coffee substitutes,NARDL,4.032052,3.229022,4.322282,0.015188,0.071391
62,"Leafy or stem vegetables, fresh or chilled",NARDL,3.873130,3.229022,4.322282,0.019313,0.086352
76,"Other sugar, confectionery and dessert",NARDL,3.827146,3.229022,4.322282,0.020695,0.091169
87,Vegetable oils,NARDL,3.424736,3.229022,4.322282,0.037563,0.144229
86,Tubers,NARDL,3.269828,3.229022,4.322282,0.047022,0.170619


### Initial Bounds-Test Results

The symmetric ARDL bounds tests identified cointegration for six food
subclasses. Thirty-six subclasses showed no evidence of cointegration, while four produced inconclusive results.

The asymmetric NARDL bounds tests identified cointegration for nine
subclasses. Thirty-one showed no evidence of cointegration, and six produced inconclusive results.

The larger number of cointegrated NARDL models suggests that separating
depreciation and appreciation may reveal long-run relationships that are concealed by the symmetric specification. However, this does not by itself establish statistically significant asymmetry.

Most food subclasses do not show evidence of a stable long-run relationship with the exchange rate under either specification. This does not rule out short-run exchange-rate effects.

The ten inconclusive models require sample-specific bounds before their final classification.

### Sample-Specific Bounds Verification

The initial bounds decisions use asymptotic critical values. These values may be less precise for the current effective sample of 99 observations.

The initial sample-specific results produced materially higher critical values than the asymptotic procedure. Consequently, all models not clearly below the asymptotic lower bound are reassessed.

This includes models initially classified as:

- cointegrated
- inconclusive

Models already below the asymptotic lower bound retain their no-cointegration classification because they do not qualify as candidate long-run relationships.

The sample-specific procedure uses 50,000 simulations and a fixed random seed to ensure reproducibility.

In [34]:
# identify candidate long-run relationships
candidate_bounds_results = (
    combined_bounds_results[
        combined_bounds_results["Decision"] != "No cointegration"
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Models requiring sample-specific verification:",
    len(candidate_bounds_results)
)

candidate_bounds_results.groupby(
    [
        "Model",
        "Decision"
    ]
).size().rename("Food_Subclasses").to_frame()

Models requiring sample-specific verification: 25


Food_Subclasses
Model Decision                      
ARDL  Cointegration                6
      Inconclusive                 4
NARDL Cointegration                9
      Inconclusive                 6

In [35]:
# calculate sample-specific bounds
exact_bounds_simulations = 50_000
exact_bounds_seed = 2026

uecm_result_collections = {
    "ARDL": symmetric_uecm_results,
    "NARDL": nardl_uecm_results
}

exact_bounds_records = []

for result_row in candidate_bounds_results.itertuples(
    index=False
):
    subclass_name = result_row.SubclassDescription
    model_name = result_row.Model

    uecm_result = uecm_result_collections[
        model_name
    ][subclass_name]

    exact_result = uecm_result.bounds_test(
        case=3,
        asymptotic=False,
        nsim=exact_bounds_simulations,
        seed=exact_bounds_seed
    )

    exact_lower_bound = exact_result.crit_vals.loc[
        95.0,
        "lower"
    ]
    exact_upper_bound = exact_result.crit_vals.loc[
        95.0,
        "upper"
    ]

    if result_row.Bounds_Statistic > exact_upper_bound:
        exact_decision = "Cointegration"
    elif result_row.Bounds_Statistic < exact_lower_bound:
        exact_decision = "No cointegration"
    else:
        exact_decision = "Inconclusive"

    exact_bounds_records.append({
        "SubclassDescription": subclass_name,
        "Model": model_name,
        "Bounds_Statistic": result_row.Bounds_Statistic,
        "Asymptotic_Decision": result_row.Decision,
        "Exact_Lower_Bound_5pct": exact_lower_bound,
        "Exact_Upper_Bound_5pct": exact_upper_bound,
        "Exact_Lower_P_Value": exact_result.p_values["lower"],
        "Exact_Upper_P_Value": exact_result.p_values["upper"],
        "Exact_Decision": exact_decision,
        "Simulations": exact_bounds_simulations
    })

exact_bounds_results = (
    pd.DataFrame(exact_bounds_records)
    .sort_values(
        [
            "Model",
            "Bounds_Statistic"
        ],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

exact_bounds_results.head(10)

,SubclassDescription,Model,Bounds_Statistic,Asymptotic_Decision,Exact_Lower_Bound_5pct,Exact_Upper_Bound_5pct,Exact_Lower_P_Value,Exact_Upper_P_Value,Exact_Decision,Simulations
0,"Chocolate, cocoa, and cocoa-based food products",ARDL,7.502411,Cointegration,5.019509,5.846901,0.006360,0.015620,Cointegration,50000
1,"Other vegetables, fresh or chilled",ARDL,7.372386,Cointegration,5.046273,5.916126,0.008000,0.016860,Cointegration,50000
2,Yoghurt and similar products,ARDL,7.295599,Cointegration,5.046273,5.916126,0.008620,0.017920,Cointegration,50000
3,Sugar,ARDL,5.502256,Cointegration,5.046273,5.916126,0.034780,0.065500,Inconclusive,50000
4,Cereals,ARDL,5.253907,Cointegration,5.019509,5.846901,0.041860,0.076580,Inconclusive,50000
5,"Nut puree, nut butter and nut pastes",ARDL,4.880924,Cointegration,5.051138,5.897994,0.056780,0.098920,No cointegration,50000
6,Pulses,ARDL,4.786498,Inconclusive,5.005447,5.885506,0.059500,0.107320,No cointegration,50000
7,Coffee and coffee substitutes,ARDL,4.413718,Inconclusive,5.019509,5.846901,0.078860,0.133040,No cointegration,50000
8,"Dates, figs and tropical fruits, fresh",ARDL,4.380068,Inconclusive,5.019509,5.846901,0.080560,0.135860,No cointegration,50000
9,"Other fruits, fresh",ARDL,3.881242,Inconclusive,5.019509,5.846901,0.117300,0.189440,No cointegration,50000


In [36]:
# summarise the sample-specific decisions
exact_bounds_decision_summary = (
    exact_bounds_results
    .groupby(
        [
            "Model",
            "Exact_Decision"
        ]
    )
    .size()
    .rename("Food_Subclasses")
    .reset_index()
)

exact_bounds_decision_summary

,Model,Exact_Decision,Food_Subclasses
0,ARDL,Cointegration,3
1,ARDL,Inconclusive,2
2,ARDL,No cointegration,5
3,NARDL,Cointegration,6
4,NARDL,Inconclusive,5
5,NARDL,No cointegration,4


In [37]:
# identify decisions changed by finite-sample verification
bounds_decision_changes = (
    exact_bounds_results[
        exact_bounds_results["Exact_Decision"]
        != exact_bounds_results["Asymptotic_Decision"]
    ]
    [
        [
            "SubclassDescription",
            "Model",
            "Bounds_Statistic",
            "Asymptotic_Decision",
            "Exact_Lower_Bound_5pct",
            "Exact_Upper_Bound_5pct",
            "Exact_Decision"
        ]
    ]
    .reset_index(drop=True)
)

bounds_decision_changes

,SubclassDescription,Model,Bounds_Statistic,Asymptotic_Decision,Exact_Lower_Bound_5pct,Exact_Upper_Bound_5pct,Exact_Decision
0,Sugar,ARDL,5.502256,Cointegration,5.046273,5.916126,Inconclusive
1,Cereals,ARDL,5.253907,Cointegration,5.019509,5.846901,Inconclusive
2,"Nut puree, nut butter and nut pastes",ARDL,4.880924,Cointegration,5.051138,5.897994,No cointegration
3,Pulses,ARDL,4.786498,Inconclusive,5.005447,5.885506,No cointegration
4,Coffee and coffee substitutes,ARDL,4.413718,Inconclusive,5.019509,5.846901,No cointegration
5,"Dates, figs and tropical fruits, fresh",ARDL,4.380068,Inconclusive,5.019509,5.846901,No cointegration
6,"Other fruits, fresh",ARDL,3.881242,Inconclusive,5.019509,5.846901,No cointegration
7,Other milk and cream,NARDL,4.766489,Cointegration,3.911175,4.963656,Inconclusive
8,Sugar,NARDL,4.686056,Cointegration,3.891827,4.994679,Inconclusive
9,"Nut puree, nut butter and nut pastes",NARDL,4.376643,Cointegration,3.911175,4.963656,Inconclusive


In [38]:
# apply exact decisions to the complete bounds table
final_bounds_results = combined_bounds_results.merge(
    exact_bounds_results[
        [
            "SubclassDescription",
            "Model",
            "Exact_Decision"
        ]
    ],
    on=[
        "SubclassDescription",
        "Model"
    ],
    how="left",
    validate="one_to_one"
)

final_bounds_results["Final_Decision"] = (
    final_bounds_results["Exact_Decision"]
    .fillna(final_bounds_results["Decision"])
)

final_bounds_results["Decision_Method"] = np.where(
    final_bounds_results["Exact_Decision"].notna(),
    "Sample-specific",
    "Asymptotic"
)

final_bounds_decision_summary = (
    final_bounds_results
    .groupby(
        [
            "Model",
            "Final_Decision"
        ]
    )
    .size()
    .rename("Food_Subclasses")
    .reset_index()
)

final_bounds_decision_summary

,Model,Final_Decision,Food_Subclasses
0,ARDL,Cointegration,3
1,ARDL,Inconclusive,2
2,ARDL,No cointegration,41
3,NARDL,Cointegration,6
4,NARDL,Inconclusive,5
5,NARDL,No cointegration,35


### Final Bounds-Test Results

Sample-specific verification materially changed the initial bounds-test
results.

For the symmetric ARDL models, three food subclasses retained evidence of cointegration. Two remained inconclusive, while 41 showed no evidence of a long-run relationship.

For the asymmetric NARDL models, six subclasses retained evidence of
cointegration. Five remained inconclusive, while 35 showed no evidence of a long-run relationship.

The sample-specific results are treated as the final bounds classifications because they account for the effective sample size of the estimated models.

Long-run pass-through coefficients will be interpreted only for models with confirmed cointegration. Models classified as inconclusive will be reported separately, while models without cointegration will not receive a long-run interpretation.

The absence of cointegration does not rule out temporary short-run
exchange-rate effects.

In [39]:
# list confirmed long-run relationships
confirmed_long_run_models = (
    final_bounds_results[
        final_bounds_results["Final_Decision"]
        == "Cointegration"
    ]
    [
        [
            "SubclassDescription",
            "Model",
            "Bounds_Statistic",
            "Decision_Method"
        ]
    ]
    .sort_values(
        [
            "Model",
            "SubclassDescription"
        ]
    )
    .reset_index(drop=True)
)

confirmed_long_run_models

,SubclassDescription,Model,Bounds_Statistic,Decision_Method
0,"Chocolate, cocoa, and cocoa-based food products",ARDL,7.502411,Sample-specific
1,"Other vegetables, fresh or chilled",ARDL,7.372386,Sample-specific
2,Yoghurt and similar products,ARDL,7.295599,Sample-specific
3,Cereals,NARDL,6.620570,Sample-specific
4,"Chocolate, cocoa, and cocoa-based food products",NARDL,6.235160,Sample-specific
5,"Dates, figs and tropical fruits, fresh",NARDL,9.173770,Sample-specific
6,"Fruit-bearing vegetables, fresh or chilled",NARDL,8.346412,Sample-specific
7,"Other vegetables, fresh or chilled",NARDL,7.406674,Sample-specific
8,Yoghurt and similar products,NARDL,7.181472,Sample-specific


In [40]:
# list inconclusive long-run relationships
inconclusive_long_run_models = (
    final_bounds_results[
        final_bounds_results["Final_Decision"]
        == "Inconclusive"
    ]
    [
        [
            "SubclassDescription",
            "Model",
            "Bounds_Statistic",
            "Decision_Method"
        ]
    ]
    .sort_values(
        [
            "Model",
            "SubclassDescription"
        ]
    )
    .reset_index(drop=True)
)

inconclusive_long_run_models

,SubclassDescription,Model,Bounds_Statistic,Decision_Method
0,Cereals,ARDL,5.253907,Sample-specific
1,Sugar,ARDL,5.502256,Sample-specific
2,Coffee and coffee substitutes,NARDL,4.032052,Sample-specific
3,Milk,NARDL,4.057681,Sample-specific
4,"Nut puree, nut butter and nut pastes",NARDL,4.376643,Sample-specific
5,Other milk and cream,NARDL,4.766489,Sample-specific
6,Sugar,NARDL,4.686056,Sample-specific


## Long-Run Coefficients and Error Correction

Long-run coefficients are extracted only for food subclasses with confirmed cointegration.

The unrestricted error-correction results contain two related sets of
coefficients:

1. the unnormalised UECM coefficients, including the lagged food-price
   coefficient that measures the speed of adjustment
2. the normalised cointegrating coefficients that describe the long-run
   relationship

Before extracting results for all confirmed subclasses, one symmetric and one asymmetric model are inspected to verify the coefficient names and scaling.

In [41]:
# inspect a confirmed symmetric UECM
reference_ardl_subclass = (
    "Chocolate, cocoa, and cocoa-based food products"
)

reference_ardl_result = symmetric_uecm_results[
    reference_ardl_subclass
]

reference_ardl_parameters = pd.DataFrame({
    "Coefficient": reference_ardl_result.params,
    "Standard_Error": reference_ardl_result.bse,
    "P_Value": reference_ardl_result.pvalues
})

reference_ardl_parameters[
    reference_ardl_parameters.index.str.contains(
        "Log_CPI|Log_ExchangeRate"
    )
]

,Coefficient,Standard_Error,P_Value
Log_CPI.L1,-0.008389,0.007680,0.277874
Log_ExchangeRate.L1,0.030973,0.011604,0.009125
D.Log_ExchangeRate.L0,0.021435,0.027480,0.437555


In [42]:
# inspect the symmetric cointegrating relationship
reference_ardl_long_run = pd.DataFrame({
    "Normalised_Coefficient": reference_ardl_result.ci_params,
    "Standard_Error": reference_ardl_result.ci_bse,
    "P_Value": reference_ardl_result.ci_pvalues
})

reference_ardl_long_run[
    reference_ardl_long_run.index.str.contains(
        "Log_CPI|Log_ExchangeRate"
    )
]

,Normalised_Coefficient,Standard_Error,P_Value
Log_CPI,1.000000,0.000000,NaN
Log_ExchangeRate,-3.692289,2.397163,0.123494


In [43]:
# inspect a confirmed asymmetric UECM
reference_nardl_subclass = "Cereals"

reference_nardl_result = nardl_uecm_results[
    reference_nardl_subclass
]

reference_nardl_parameters = pd.DataFrame({
    "Coefficient": reference_nardl_result.params,
    "Standard_Error": reference_nardl_result.bse,
    "P_Value": reference_nardl_result.pvalues
})

reference_nardl_parameters[
    reference_nardl_parameters.index.str.contains(
        "Log_CPI|ExchangeRate"
    )
]

,Coefficient,Standard_Error,P_Value
Log_CPI.L1,-0.066162,0.020543,0.001850
ExchangeRate_Positive_Cumulative_Pct.L1,0.000881,0.000217,0.000117
ExchangeRate_Negative_Cumulative_Pct.L1,0.000935,0.000277,0.001148
D.ExchangeRate_Positive_Cumulative_Pct.L0,-0.001090,0.000753,0.151502
D.ExchangeRate_Positive_Cumulative_Pct.L1,-0.002766,0.000794,0.000809
D.ExchangeRate_Negative_Cumulative_Pct.L0,-0.000012,0.001133,0.991817
D.ExchangeRate_Negative_Cumulative_Pct.L1,0.001768,0.001084,0.106715


In [44]:
# inspect the asymmetric cointegrating relationship
reference_nardl_long_run = pd.DataFrame({
    "Normalised_Coefficient": reference_nardl_result.ci_params,
    "Standard_Error": reference_nardl_result.ci_bse,
    "P_Value": reference_nardl_result.ci_pvalues
})

reference_nardl_long_run[
    reference_nardl_long_run.index.str.contains(
        "Log_CPI|ExchangeRate"
    )
]

,Normalised_Coefficient,Standard_Error,P_Value
Log_CPI,1.000000,0.000000,NaN
ExchangeRate_Positive_Cumulative_Pct,-0.013309,0.004275,0.001851
ExchangeRate_Negative_Cumulative_Pct,-0.014136,0.005860,0.015854


### Reference-Model Coefficient Interpretation

The reference ARDL model for chocolate products has a negative adjustment coefficient, but it is not statistically significant at the 5% level. Its normalised long-run exchange-rate coefficient is also imprecisely estimated.

This demonstrates that a confirmed bounds test should be considered together with the sign and significance of the error-correction coefficient before the long-run multiplier is interpreted.

The reference NARDL model for cereals has a negative and statistically
significant adjustment coefficient. Approximately 6.6% of the previous
month's deviation from long-run equilibrium is corrected during the current month.

The cereals model therefore provides a clearer example of a stable
error-correction relationship. Its depreciation and appreciation coefficients must still be formally compared before concluding that pass-through is asymmetric.